# Image Captioning — Flickr8k + Xception + LSTM
**With BLEU score evaluation + Gradio live demo**

> Make sure to switch runtime to GPU before running: Runtime > Change runtime type > T4 GPU

## Step 1 — Install & Download

In [ ]:
!pip install gradio nltk -q

# grab the Flickr8k dataset from Kaggle
from google.colab import files
print("Upload your kaggle.json (Kaggle > Account > Create New Token):")
files.upload()
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d adityajn105/flickr8k -p /content/flickr8k --unzip -q
print("Done - Flickr8k ready")

## Step 2 — Imports

In [ ]:
import os, string, random, glob
from itertools import chain
from pickle import dump, load
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image
from tqdm.notebook import tqdm
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
from nltk.translate.bleu_score import corpus_bleu
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical, plot_model
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Input
from tensorflow.keras.applications.xception import Xception
from tensorflow.keras.models import Model, load_model
from keras.layers import add
from IPython.display import Image as IPImage
print("Imports OK")

## Step 3 — Load & clean captions

In [ ]:
caption_candidates = glob.glob("/content/flickr8k/**/*.txt", recursive=True)
caption_file = [f for f in caption_candidates if "caption" in f.lower() or "token" in f.lower()]
try:
    CAPTIONS_FILE = caption_file[0]
    print("Using:", CAPTIONS_FILE)

    def load_captions(filepath):
        dataset = {}
        with open(filepath, "r") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split("\t") if "\t" in line else line.split(",", 1)
                if len(parts) < 2:
                    continue
                img_name = parts[0].split("#")[0].strip()
                caption  = parts[1]
                caption  = caption.translate(str.maketrans("", "", string.punctuation))
                caption  = caption.replace("-", " ").lower().strip()
                caption  = "<start> " + " ".join(caption.split()) + " <end>"
                dataset.setdefault(img_name, []).append(caption)
        return dataset

    dataset = load_captions(CAPTIONS_FILE)
    print(f"Loaded captions for {len(dataset)} images")
    sample = list(dataset.keys())[0]
    print("Sample:", dataset[sample][0])

except IndexError:
    print("Error: No caption files were found. This often happens if the Flickr8k dataset was not downloaded or unzipped correctly.")
    print("Please check the output of the previous cell (Step 1) to ensure the dataset was downloaded successfully.")
    print("You might need to re-upload 'kaggle.json' and re-run Step 1.")
    CAPTIONS_FILE = None
    dataset = {}
except Exception as e:
    print(f"An unexpected error occurred: {e}")

## Step 4 — Train / test split

In [ ]:
img_dirs = (glob.glob("/content/flickr8k/**/Images", recursive=True) +
            glob.glob("/content/flickr8k/**/images", recursive=True))
IMAGES_DIR = img_dirs[0] if img_dirs else "/content/flickr8k"
print("Images dir:", IMAGES_DIR)

available  = set(os.listdir(IMAGES_DIR))
valid_keys = [k for k in dataset if k in available]
print(f"{len(valid_keys)} images found on disk")

random.seed(42)
random.shuffle(valid_keys)
split         = int(0.9 * len(valid_keys))
train_keys    = valid_keys[:split]
test_keys     = valid_keys[split:]
train_dataset = {k: dataset[k] for k in train_keys}
test_dataset  = {k: dataset[k] for k in test_keys}
print(f"Train: {len(train_dataset)} | Test: {len(test_dataset)}")

## Step 5 — Visualise sample images

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for ax, key in zip(axes.flat, random.sample(train_keys, 12)):
    img = Image.open(os.path.join(IMAGES_DIR, key))
    ax.imshow(img)
    ax.axis("off")
    preview = dataset[key][0].replace("<start> ","").replace(" <end>","")[:50]
    ax.set_title(preview, fontsize=8)
plt.suptitle("Sample Flickr8k Training Images", fontsize=14)
plt.tight_layout()
plt.show()

## Step 6 — Tokenizer

In [ ]:
all_captions = list(chain.from_iterable(train_dataset.values()))
tokenizer = Tokenizer(oov_token="<oov>")
tokenizer.fit_on_texts(all_captions)
total_words = len(tokenizer.word_index) + 1

def get_max_length(descriptions):
    return max(len(d.split()) for caps in descriptions.values() for d in caps)

max_length = get_max_length(train_dataset)
print(f"Vocabulary: {total_words} | Max caption length: {max_length}")

## Step 7 — Extract Xception features
> Heads up — this takes around 8 minutes on GPU. Features get saved to disk so you don't have to rerun it.

In [ ]:
def extract_feature(img_path, model):
    try:
        img = Image.open(img_path).resize((299, 299)).convert("RGB")
    except:
        return None
    arr = np.expand_dims(np.array(img) / 127.5 - 1.0, axis=0)
    return model.predict(arr, verbose=0)

xception_model = Xception(include_top=False, pooling="avg")

image_features = {}
for key in tqdm(valid_keys, desc="Extracting features"):
    feat = extract_feature(os.path.join(IMAGES_DIR, key), xception_model)
    if feat is not None:
        image_features[key] = feat

with open("/content/image_features.pkl", "wb") as f:
    dump(image_features, f)
print(f"Saved features for {len(image_features)} images")

## Step 8 — Define model

In [ ]:
def create_sequences(tokenizer, max_length, desc_list, feature):
    X1, X2, y = [], [], []
    for desc in desc_list:
        seq = tokenizer.texts_to_sequences([desc])[0]
        for i in range(1, len(seq)):
            in_seq  = pad_sequences([seq[:i]], maxlen=max_length)[0]
            out_seq = to_categorical([seq[i]], num_classes=total_words)[0]
            X1.append(feature)
            X2.append(in_seq)
            y.append(out_seq)
    return np.array(X1), np.array(X2), np.array(y)


def data_generator(descriptions, features, tokenizer, max_length):
    while True:
        for key, desc_list in descriptions.items():
            if key not in features:
                continue
            X1, X2, y = create_sequences(tokenizer, max_length, desc_list, features[key][0])
            yield ([X1, X2], y)


def define_model(total_words, max_length):
    inputs1 = Input(shape=(2048,))
    fe1     = Dropout(0.5)(inputs1)
    fe2     = Dense(256, activation="relu")(fe1)

    inputs2 = Input(shape=(max_length,))
    se1     = Embedding(total_words, 256, mask_zero=True)(inputs2)
    se2     = Dropout(0.5)(se1)
    se3     = LSTM(256, use_cudnn=False)(se2)

    decoder1 = add([fe2, se3])
    decoder2 = Dense(256, activation="relu")(decoder1)
    outputs  = Dense(total_words, activation="softmax")(decoder2)

    model = Model(inputs=[inputs1, inputs2], outputs=outputs)
    model.compile(loss="categorical_crossentropy", optimizer="adam")
    model.summary()
    return model

model = define_model(total_words, max_length)
plot_model(model, to_file="model.png", show_shapes=True)
IPImage("model.png")

## Step 9 — Train
> This takes about 25 minutes on a T4 GPU, so grab a coffee.

In [ ]:
EPOCHS = 20
steps  = len(train_dataset)
os.makedirs("/content/models", exist_ok=True)

# keras expects a tuple of inputs, not a list, so I wrap the generator to fix that
def _corrected_data_generator_output(original_gen):
    for item in original_gen:
        # the original generator spits out ([X1, X2], y)
        inputs_list, target = item
        # convert the inputs list to a tuple so keras doesn't complain
        yield (tuple(inputs_list), target)

# need to tell tf.data what the output types and shapes look like
# X1 is the image feature vector — shape (batch, 2048), float32
# X2 is the padded token sequence — shape (batch, max_length), int32
# y is the one-hot target word — shape (batch, total_words), float32
output_types = ((tf.float32, tf.int32), tf.float32)
output_shapes = ((tf.TensorShape([None, 2048]), tf.TensorShape([None, max_length])), tf.TensorShape([None, total_words]))

for epoch in range(EPOCHS):
    # create a fresh generator for each epoch
    gen = data_generator(train_dataset, image_features, tokenizer, max_length)

    # wrap it so the inputs come out as a tuple
    corrected_gen = _corrected_data_generator_output(gen)

    # wrap in tf.data so we can use .take()
    dataset = tf.data.Dataset.from_generator(
        lambda: corrected_gen,
        output_types=output_types,
        output_shapes=output_shapes
    )
    # generator loops forever, so .take(steps) limits it to one epoch
    dataset = dataset.take(steps)

    model.fit(dataset, epochs=2, verbose=1)
    model.save(f"/content/models/model_{epoch}.h5")
    print(f"Epoch {epoch+1}/{EPOCHS} saved")

## Step 10 — BLEU Score
> BLEU tells you how close your generated captions are to the real ones. Ranges from 0 to 1, higher is better.
> A BLEU-1 of 0.5 basically means half the words in my output matched the reference captions.

In [ ]:
def word_for_id(integer, tokenizer):
    for word, index in tokenizer.word_index.items():
        if index == integer:
            return word
    return None


def generate_caption(model, tokenizer, photo, max_length):
    in_text = "<start>"
    end_token_id = tokenizer.word_index.get("<end>") # Get the ID for the special <end> token

    for _ in range(max_length):
        seq = pad_sequences([tokenizer.texts_to_sequences([in_text])[0]], maxlen=max_length)
        pred_id = np.argmax(model.predict([photo, seq], verbose=0)) # argmax gives the most likely next word ID

        word = word_for_id(pred_id, tokenizer) # convert the ID back to an actual word

        if word is None: # shouldn't happen, but bail out if the ID doesn't map to anything
            break

        # If special <end> token ID is predicted, or the word 'end' itself is predicted, break without appending it
        if pred_id == end_token_id or word == 'end':
            break

        in_text += " " + word # only tack on real words<end>

    return in_text.replace("<start>", "").strip()


# rebuild the model architecture before loading weights
best_model = define_model(total_words, max_length)

# now load the saved weights from the last epoch
best_model.load_weights(f"/content/models/model_{EPOCHS-1}.h5")

actual, predicted = [], []
for key in tqdm(test_keys[:200], desc="Evaluating BLEU"):
    if key not in image_features:
        continue
    refs = [cap.replace("<start>","").replace("<end>","").split() for cap in test_dataset[key]]
    gen  = generate_caption(best_model, tokenizer, image_features[key], max_length).split()
    actual.append(refs)
    predicted.append(gen)

bleu1 = corpus_bleu(actual, predicted, weights=(1,0,0,0))
bleu2 = corpus_bleu(actual, predicted, weights=(0.5,0.5,0,0))
bleu3 = corpus_bleu(actual, predicted, weights=(0.33,0.33,0.33,0))
bleu4 = corpus_bleu(actual, predicted, weights=(0.25,0.25,0.25,0.25))

print(f"BLEU-1: {bleu1:.3f}")
print(f"BLEU-2: {bleu2:.3f}")
print(f"BLEU-3: {bleu3:.3f}")
print(f"BLEU-4: {bleu4:.3f}")

## Step 11 — View sample predictions

In [ ]:
sample_tests = random.sample([k for k in test_keys if k in image_features], 5)

for key in sample_tests:
    caption = generate_caption(best_model, tokenizer, image_features[key], max_length)
    img = Image.open(os.path.join(IMAGES_DIR, key))
    ref = test_dataset[key][0].replace("<start> ","").replace(" <end>","")
    plt.figure(figsize=(8, 5))
    plt.imshow(img)
    plt.title(f"Generated: {caption}\nReference: {ref}", fontsize=10, wrap=True)
    plt.axis("off")
    plt.tight_layout()
    plt.show()
    print(f"Generated : {caption}")
    print(f"Reference : {ref}")
    print("-"*60)

## Step 12 — Gradio Live Demo
> Spins up a live web app where you can upload any image and get a caption back. Setting share=True generates a public link that lasts 72 hours — handy for demos.

In [ ]:
import gradio as gr
import random

def caption_image(input_image):
    img = input_image.resize((299, 299)).convert("RGB")
    arr = np.expand_dims(np.array(img) / 127.5 - 1.0, axis=0)
    feature = xception_model.predict(arr, verbose=0)
    return generate_caption(best_model, tokenizer, feature, max_length)

demo = gr.Interface(
    fn=caption_image,
    inputs=gr.Image(type="pil", label="Upload an image"),
    outputs=gr.Textbox(label="Generated Caption"),
    title="Image Captioning — Xception + LSTM",
    description="Upload any image and the model will generate a caption.",
    examples=[[os.path.join(IMAGES_DIR, k)] for k in random.sample(test_keys[:20], 3)]
)

demo.launch(share=True)